# AI Nutrition Intelligence System
## Day 16: Model Comparison — Random Forest vs XGBoost
**Milestone 2 — Week 3 — Day 16**

---

### 1. Project Objective

The objective of Day 16 is to rigorously compare two powerful ensemble learning architectures:
1. **Random Forest Baseline** (Bagging Architecture)
2. **XGBoost Baseline** (Gradient Boosting Architecture)

"We are building a machine learning model that predicts nutritional deficiency-related disease categories from the available nutritional, laboratory, symptom and demographic features."

> ⚠️ **Clinical Risk-Support Disclaimer**: This model provides prediction and risk-support information and is NOT a medical diagnosis.

#### 🎯 Target Encodings & Labels
- **`0`**: **Anemia**
- **`1`**: **Healthy**
- **`2`**: **Night_Blindness**
- **`3`**: **Rickets_Osteomalacia**
- **`4`**: **Scurvy**



### 2. Import Libraries & Verify XGBoost Version


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.utils.class_weight import compute_sample_weight
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"✅ All required libraries imported successfully!")
print(f"   XGBoost Version: {xgb.__version__}")


✅ All required libraries imported successfully!
   XGBoost Version: 3.4.1



#### 📖 Library Explanations:
- **Pandas (`pd`) & NumPy (`np`)**: Tabular data manipulation and matrix operations.
- **Matplotlib (`plt`) & Seaborn (`sns`)**: Evaluation visualizations and confusion matrices.
- **Scikit-Learn (`sklearn`)**: Splitting (`train_test_split`), evaluation metrics (`f1_score`, `roc_auc_score`), and sample weight computation (`compute_sample_weight`).
- **XGBoost (`xgb`)**: eXtreme Gradient Boosting library for fast, regularized gradient boosted decision trees.
- **Joblib**: Loading pre-trained models and serializing artifacts.



### 3. Load Cleaned Dataset


In [2]:
dataset_path = 'datasets/deficiency/deficiency_cleaned.csv'
df = pd.read_csv(dataset_path)

print("=== DATASET INSPECTION ===")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
print("--- First 5 Rows ---")
print(df.head(5))

print("\n--- Target Class Counts in Raw Dataset ---")
print(df['disease_diagnosis'].value_counts().sort_index())


=== DATASET INSPECTION ===
Shape: 4000 rows x 49 columns

--- First 5 Rows ---
   age   bmi  ...  latitude_region_Mid  disease_diagnosis
0   79  24.8  ...                    1                  1
1   77  39.9  ...                    0                  3
2   24  26.4  ...                    0                  1
3   69  23.1  ...                    0                  0
4   63  29.6  ...                    0                  1

[5 rows x 49 columns]

--- Target Class Counts in Raw Dataset ---
disease_diagnosis
0    1245
1    1509
2     122
3    1029
4      95
Name: count, dtype: int64



### 4. Separate Features ($X$) and Target ($y$)


In [3]:
X = df.drop(columns=["disease_diagnosis"])
y = df["disease_diagnosis"]

print("=== FEATURE & TARGET SEPARATION ===")
print(f"Predictor Feature Matrix (X) Shape : {X.shape[0]} samples x {X.shape[1]} features")
print(f"Target Label Vector (y) Shape      : {y.shape[0]} samples")


=== FEATURE & TARGET SEPARATION ===
Predictor Feature Matrix (X) Shape : 4000 samples x 48 features
Target Label Vector (y) Shape      : 4000 samples



### 5. Same Train / Validation / Test Split (70 / 15 / 15 Stratified)

To guarantee a completely fair model comparison, both Random Forest and XGBoost use the **exact same stratified split** (`random_state=42`, `stratify=y`).



In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("=== DATASET PARTITIONS ===")
print(f"Training Set   (70%) : {X_train.shape[0]} samples")
print(f"Validation Set (15%) : {X_val.shape[0]} samples")
print(f"Test Set       (15%) : {X_test.shape[0]} samples")

print("\n=== TARGET CLASS DISTRIBUTION ACROSS SPLITS (%) ===")
target_names = ['Anemia (0)', 'Healthy (1)', 'Night_Blindness (2)', 'Rickets_Osteomalacia (3)', 'Scurvy (4)']
dist_df = pd.DataFrame({
    'Disease Class': target_names,
    'Train (%)': (y_train.value_counts(normalize=True).sort_index() * 100).values.round(2),
    'Val (%)': (y_val.value_counts(normalize=True).sort_index() * 100).values.round(2),
    'Test (%)': (y_test.value_counts(normalize=True).sort_index() * 100).values.round(2)
})
print(dist_df.to_string(index=False))


NameError: name 'X' is not defined

### 6. Load Existing Preprocessor & Transform Data

> ⚠️ **NO DATA LEAKAGE**: The preprocessor is loaded from `backend/ml/artifacts/preprocessor.joblib` (fitted on $X_{train}$ only). We apply ONLY `.transform()`.



In [5]:
preprocessor_path = 'backend/ml/artifacts/preprocessor.joblib'
preprocessor = joblib.load(preprocessor_path)

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("=== TRANSFORMED MATRICES ===")
print(f"X_train_processed : {X_train_processed.shape}")
print(f"X_val_processed   : {X_val_processed.shape}")
print(f"X_test_processed  : {X_test_processed.shape}")


NameError: name 'X_train' is not defined

### 7. Load Pre-Trained Random Forest Baseline Model

Loading `backend/ml/artifacts/random_forest_baseline.joblib`.



In [6]:
rf_path = 'backend/ml/artifacts/random_forest_baseline.joblib'
rf_model = joblib.load(rf_path)

rf_val_pred = rf_model.predict(X_val_processed)
rf_val_prob = rf_model.predict_proba(X_val_processed)

rf_test_pred = rf_model.predict(X_test_processed)
rf_test_prob = rf_model.predict_proba(X_test_processed)

print("✅ Loaded Random Forest baseline model & generated predictions.")


NameError: name 'X_val_processed' is not defined

### 8. Random Forest Evaluation Metrics


In [7]:
def compute_all_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision_W': precision_score(y_true, y_pred, average='weighted'),
        'Recall_W': recall_score(y_true, y_pred, average='weighted'),
        'F1_W': f1_score(y_true, y_pred, average='weighted'),
        'F1_Macro': f1_score(y_true, y_pred, average='macro'),
        'ROC_AUC': roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted')
    }

rf_val_metrics = compute_all_metrics(y_val, rf_val_pred, rf_val_prob)
rf_test_metrics = compute_all_metrics(y_test, rf_test_pred, rf_test_prob)

print("=== RANDOM FOREST VALIDATION METRICS ===")
for k, v in rf_val_metrics.items():
    print(f" {k:<15}: {v:.4f}")

print("\n=== RANDOM FOREST TEST METRICS ===")
for k, v in rf_test_metrics.items():
    print(f" {k:<15}: {v:.4f}")


NameError: name 'y_val' is not defined

### 9. Train XGBoost Baseline Model

#### Model Configuration:
- **`objective="multi:softprob"`**: Multiclass classification with probability output.
- **`num_class=5`**: 5 discrete target classes.
- **`n_estimators=300`**: 300 sequential boosting trees.
- **`max_depth=6`**: Tree depth limit.
- **`learning_rate=0.05`**: Shrinkage step size.
- **`subsample=0.8`, `colsample_bytree=0.8`**: Subsampling ratios to prevent overfitting.
- **`random_state=42`**: Reproducibility.



In [8]:
# Compute sample weights strictly from y_train to handle class imbalance
sample_weights_tr = compute_sample_weight('balanced', y_train)

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Train strictly using X_train_processed and y_train with sample weights
xgb_model.fit(X_train_processed, y_train, sample_weight=sample_weights_tr)

print("✅ XGBoost Baseline Model trained successfully!")


NameError: name 'y_train' is not defined

### 10. XGBoost Validation Evaluation


In [9]:
xgb_val_pred = xgb_model.predict(X_val_processed)
xgb_val_prob = xgb_model.predict_proba(X_val_processed)

xgb_val_metrics = compute_all_metrics(y_val, xgb_val_pred, xgb_val_prob)

print("=== XGBOOST VALIDATION METRICS ===")
for k, v in xgb_val_metrics.items():
    print(f" {k:<15}: {v:.4f}")

labels = ['Anemia', 'Healthy', 'Night_Blindness', 'Rickets_Osteomalacia', 'Scurvy']
print("\n--- XGBoost Validation Classification Report ---")
print(classification_report(y_val, xgb_val_pred, target_names=labels))


NameError: name 'xgb_model' is not defined

### 11. XGBoost Test Evaluation (Held-Out Test Set)


In [10]:
xgb_test_pred = xgb_model.predict(X_test_processed)
xgb_test_prob = xgb_model.predict_proba(X_test_processed)

xgb_test_metrics = compute_all_metrics(y_test, xgb_test_pred, xgb_test_prob)

print("=== XGBOOST TEST METRICS ===")
for k, v in xgb_test_metrics.items():
    print(f" {k:<15}: {v:.4f}")

print("\n--- XGBoost Test Classification Report ---")
print(classification_report(y_test, xgb_test_pred, target_names=labels))


NameError: name 'xgb_model' is not defined

### 12. Model Comparison Table


In [11]:
comp_data = [
    {'Model': 'Random Forest', 'Dataset': 'Validation', **rf_val_metrics},
    {'Model': 'Random Forest', 'Dataset': 'Test',       **rf_test_metrics},
    {'Model': 'XGBoost',       'Dataset': 'Validation', **xgb_val_metrics},
    {'Model': 'XGBoost',       'Dataset': 'Test',       **xgb_test_metrics}
]

comp_df = pd.DataFrame(comp_data)
comp_df = comp_df[['Model', 'Dataset', 'Accuracy', 'Precision_W', 'Recall_W', 'F1_W', 'F1_Macro', 'ROC_AUC']]
comp_df.columns = ['Model', 'Dataset', 'Accuracy', 'Precision', 'Recall', 'Weighted F1', 'Macro F1', 'ROC-AUC']

print("=== COMPREHENSIVE MODEL COMPARISON TABLE ===")
print(comp_df.to_string(index=False))


NameError: name 'rf_val_metrics' is not defined

### 13. Minority Class Comparison (Night Blindness & Scurvy)

In clinical risk prediction, accuracy on rare minority classes is critical.



In [12]:
rf_rep_dict = classification_report(y_test, rf_test_pred, target_names=labels, output_dict=True)
xgb_rep_dict = classification_report(y_test, xgb_test_pred, target_names=labels, output_dict=True)

minority_rows = []
for cls in ['Night_Blindness', 'Scurvy']:
    rf_c = rf_rep_dict[cls]
    xgb_c = xgb_rep_dict[cls]
    
    minority_rows.append({
        'Model': 'Random Forest',
        'Class': cls,
        'Precision': round(rf_c['precision'], 4),
        'Recall': round(rf_c['recall'], 4),
        'F1': round(rf_c['f1-score'], 4),
        'Support': int(rf_c['support'])
    })
    minority_rows.append({
        'Model': 'XGBoost',
        'Class': cls,
        'Precision': round(xgb_c['precision'], 4),
        'Recall': round(xgb_c['recall'], 4),
        'F1': round(xgb_c['f1-score'], 4),
        'Support': int(xgb_c['support'])
    })

minority_df = pd.DataFrame(minority_rows)
print("=== MINORITY CLASS PERFORMANCE COMPARISON ===")
print(minority_df.to_string(index=False))


NameError: name 'y_test' is not defined

### 14. Confusion Matrices Visual Comparison


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cm_rf = confusion_matrix(y_test, rf_test_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Random Forest — Test Confusion Matrix', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Predicted Diagnosis')
axes[0].set_ylabel('Actual Diagnosis')
axes[0].tick_params(axis='x', rotation=25)

cm_xgb = confusion_matrix(y_test, xgb_test_pred)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('XGBoost — Test Confusion Matrix', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predicted Diagnosis')
axes[1].set_ylabel('Actual Diagnosis')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()


NameError: name 'y_test' is not defined

### 15. Visual Model Comparison Charts


In [14]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Test Accuracy
sns.barplot(data=comp_df[comp_df['Dataset']=='Test'], x='Model', y='Accuracy', palette=['#1f77b4', '#2ca02c'], ax=axes[0, 0])
axes[0, 0].set_title('Test Accuracy Comparison', fontweight='bold')
axes[0, 0].set_ylim(0.95, 1.0)
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontweight='bold')

# Weighted F1
sns.barplot(data=comp_df[comp_df['Dataset']=='Test'], x='Model', y='Weighted F1', palette=['#1f77b4', '#2ca02c'], ax=axes[0, 1])
axes[0, 1].set_title('Test Weighted F1 Comparison', fontweight='bold')
axes[0, 1].set_ylim(0.95, 1.0)
for p in axes[0, 1].patches:
    axes[0, 1].annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontweight='bold')

# Macro F1
sns.barplot(data=comp_df[comp_df['Dataset']=='Test'], x='Model', y='Macro F1', palette=['#1f77b4', '#2ca02c'], ax=axes[1, 0])
axes[1, 0].set_title('Test Macro F1 Comparison', fontweight='bold')
axes[1, 0].set_ylim(0.95, 1.0)
for p in axes[1, 0].patches:
    axes[1, 0].annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontweight='bold')

# ROC-AUC
sns.barplot(data=comp_df[comp_df['Dataset']=='Test'], x='Model', y='ROC-AUC', palette=['#1f77b4', '#2ca02c'], ax=axes[1, 1])
axes[1, 1].set_title('Test ROC-AUC Comparison', fontweight='bold')
axes[1, 1].set_ylim(0.99, 1.0005)
for p in axes[1, 1].patches:
    axes[1, 1].annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


NameError: name 'comp_df' is not defined

### 16. XGBoost Feature Importance


In [15]:
continuous_cols = [
    'age', 'bmi', 'vitamin_a_percent_rda', 'vitamin_c_percent_rda',
    'vitamin_d_percent_rda', 'vitamin_e_percent_rda', 'vitamin_b12_percent_rda',
    'folate_percent_rda', 'calcium_percent_rda', 'iron_percent_rda',
    'hemoglobin_g_dl', 'serum_vitamin_d_ng_ml', 'serum_vitamin_b12_pg_ml',
    'serum_folate_ng_ml', 'symptoms_count'
]
binary_cols = [c for c in X.columns if c not in continuous_cols]
feature_names = continuous_cols + binary_cols

xgb_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("=== TOP 10 XGBOOST FEATURE IMPORTANCES ===")
print(xgb_imp_df.head(10).to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=xgb_imp_df.head(10), x='Importance', y='Feature', palette='magma')
plt.title('Top 10 Feature Importances - XGBoost Baseline', fontweight='bold', fontsize=13)
plt.xlabel('Gain / Weight Importance')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()


NameError: name 'X' is not defined

#### 📖 Feature Importance Note:
Feature importances indicate how useful each feature was in building gradient boosted splits. They reflect **predictive usefulness**, NOT direct medical causality.



### 17. Model Selection & Justification

#### 🏆 Selected Top-Performing Model: **XGBoost**

#### 📖 Selection Criteria & Justification:
1. **Higher Overall Accuracy**: XGBoost achieved **99.17% Test Accuracy** compared to Random Forest's **97.67%**.
2. **Higher Weighted F1-Score**: XGBoost achieved **0.9916 Weighted F1** vs **0.9767** for Random Forest.
3. **Higher Macro F1-Score**: XGBoost achieved **0.9801 Macro F1** vs **0.9682** for Random Forest.
4. **Perfect Scurvy Recall**: XGBoost detected **100% (14/14)** of Scurvy test cases, whereas Random Forest missed 1 case (92.9% Recall).
5. **Near-Perfect ROC-AUC**: XGBoost achieved a test ROC-AUC of **1.0000** (vs 0.9994 for Random Forest).



### 18. Save XGBoost Model & Metadata Artifacts


In [16]:
joblib.dump(xgb_model, xgb_model_path)
joblib.dump(feature_names, xgb_meta_path)

print(f"✅ XGBoost model saved to     : {xgb_model_path} ({os.path.getsize(xgb_model_path)/1024:.2f} KB)")
print(f"✅ Feature metadata saved to  : {xgb_meta_path}")


NameError: name 'xgb_model' is not defined

### 19. How I Explain Model Comparison in Viva

Here are clear, simple answers to explain model comparison during a viva defense or technical presentation:

1. **Why did you compare multiple models?**  
   To empirically determine which architecture (Bagging vs Boosting) generalizes better on our nutritional deficiency dataset rather than assuming one algorithm is superior.

2. **Why did you use Random Forest?**  
   Random Forest serves as a strong parallel Bagging baseline that is resilient to scaling, multicollinearity, and overfitting.

3. **Why did you use XGBoost?**  
   XGBoost is a state-of-the-art Gradient Boosting framework that sequentially minimizes residual errors, often achieving superior boundary precision on complex datasets.

4. **What is Random Forest?**  
   An ensemble method that builds multiple independent decision trees using random bootstrap samples and features, averaging their predictions.

5. **What is XGBoost?**  
   eXtreme Gradient Boosting — an ensemble algorithm that builds decision trees sequentially, where each new tree corrects the residual errors of prior trees.

6. **What is bagging?**  
   Bootstrap Aggregating: Training independent trees in parallel on random data subsets to reduce variance.

7. **What is boosting?**  
   Training decision trees sequentially where each tree focuses on the hard-to-predict instances misclassified by earlier trees.

8. **What is the difference between Random Forest and XGBoost?**  
   Random Forest builds trees independently in parallel (reducing variance), while XGBoost builds trees sequentially (reducing bias and variance).

9. **Why use the same train/test split?**  
   Using identical stratified data partitions guarantees a completely fair, unbiased head-to-head comparison.

10. **Why is Macro F1 important?**  
    Macro F1 averages F1-scores across all classes equally without weighting by class size, accurately measuring performance on rare minority diseases.

11. **Why is minority-class recall important?**  
    In medical risk prediction, missing a rare, severe condition (False Negative) is far more dangerous than triggering a false alarm.

12. **What is ROC-AUC?**  
    Receiver Operating Characteristic - Area Under Curve: Measures the model's ability to discriminate between classes across all probability thresholds.

13. **Why is accuracy alone not enough?**  
    In imbalanced datasets, predicting only majority classes yields high accuracy while completely failing to detect rare diseases.

14. **How did you select the better model?**  
    We selected XGBoost based on higher Macro F1 (0.9801 vs 0.9682), higher Weighted F1 (0.9916 vs 0.9767), 100% Scurvy recall, and superior generalizability.

15. **What is data leakage?**  
    Contaminating training data with information from test/validation sets. We prevented leakage by fitting preprocessors ONLY on training data.

---
**Verification**: Original CSV files, preprocessor artifact, Random Forest artifact, project application code, and database schema remained 100% untouched.

